### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [22]:
import os
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/05/25s4fzd90_906hkl3kjr7vxc0000gn/T/ipykernel_3878/2865240289.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader


In [23]:
# processing all the pdfs
def process_all_pdf(pdf_directory):
    """Process all the PDFs in the given directory"""
    all_docs = [] #initially empty
    pdf_dir = Path(pdf_directory)

    """ Find all the pdf files in the dir"""
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdfs in {pdf_dir} directory")

    """ Processing each file"""

    for pdf in pdf_files:
        print(f"Processing {pdf}")
        try :
            # Loading a file
            loader = PyPDFLoader(str(pdf))
            documents = loader.load()
            # adding metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf.name
                doc.metadata["file_type"] = "pdf"

            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"error {e}")

    print(f"Total number of documents loaded: {len(all_docs)}")
    return all_docs
    


all_pdf_documents = process_all_pdf("/Users/Armaan/Desktop/learn/RAG/data/pdfs")

Found 2 pdfs in /Users/Armaan/Desktop/learn/RAG/data/pdfs directory
Processing /Users/Armaan/Desktop/learn/RAG/data/pdfs/RAG_Experience.pdf
Loaded 36 pages
Processing /Users/Armaan/Desktop/learn/RAG/data/pdfs/RAG_Survey.pdf
Loaded 21 pages
Total number of documents loaded: 57


### Text Splitting into Chunks

In [24]:
def split_documents(documents, chunk_size=1000, chunks_overlap=200):
    """Split docs into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap=chunks_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents=documents)
    print(f"Split {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print("Example Chunk:")
        print("Content:")
        print(f"{split_docs[0].page_content[:200]}...")
        print("Metadata:")
        print(f"{split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

### Embedding and VectorstoreDB

In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity
import os

In [ ]:
class EmbeddingManager:
    """It will handle  the generation of document embeddings via sentence transformers"""

    def __init__(self, model_name:str= "all-MiniLM-L6-v2"):
        """Initialize the embedding manager"""

        """
        Args : Huggingface model name for sentence transformers
        """
        self.model_name=model_name
        self.model = None 
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print("Trying to load the model")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str])-> np.ndarray:
        """
        Generate embeddings for a list of the text via the model
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)}  texts...")
        embeddings= self.model.encode(texts, show_progress_bar=True)
        print(f"Generate embeddins with shape of {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


Trying to load the model


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10799.79it/s]


Model loaded successfully. Embedding Dimension: 384


### VectorStore

In [28]:
class VectorStore:
    """Manages the vector embeddings in the vector database; chromedb in our case"""

    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory=persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # create persistent chromDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                self.collection_name,
                metadata={"description" :"PDF document containing embeddings for RAG"}
                )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection :{self.collection.count()} " )
        except Exception as e:
            printf("Error while initializing the store: {e}")


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids:List[str] = []
        metadatas:List[Any] = []
        documents_text: List[Any] = []
        embeddings_list: List[List[Any]] = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise



vector_store = VectorStore()
vector_store


Vector store initialized. Collection: pdf_documents
Existing documents in collection :0 


In [34]:
texts = [ doc.page_content for doc in chunks]
texts
# storring them in chroma db
embeddings = embedding_manager.generate_embeddings(texts=texts)

vector_store.add_documents(documents=chunks,embeddings=embeddings)

Generating embeddings for 242  texts...


Batches: 100%|██████████| 8/8 [00:01<00:00,  6.39it/s]

Generate embeddins with shape of (242, 384)
Adding 242 documents to vector store...
Successfully added 242 documents to vector store
Total documents in collection: 242
